# GPT: trening, skala i analiza zachowania

## Zadanie
Zbadaj GPT jako architekturę decoder-only i procedurę treningową. Eksperyment ilustruje rolę maski causal, oszacujemy parametry i koszt attention, a następnie porównamy GPT-3, GPT-4 i ChatGPT. Szczegóły GPT-4 nie są publiczne w stopniu pozwalającym odtworzyć jego liczbę parametrów.

## Rama teoretyczna: co właściwie skalujemy w GPT?

### Model bazowy i cel predykcyjny
GPT to autoregresyjny model probabilistyczny z faktoryzacją $p_\theta(x_{1:T})=\prod_t p_\theta(x_t\mid x_{<t})$. Blok transformera aktualizuje reprezentację każdego tokenu przez causal self-attention i MLP. Podczas pretrainingu optymalizujemy ujemną log-wiarygodność tekstu. Architektura wyznacza rodzinę funkcji, natomiast dane, optymalizator, harmonogram uczenia i cel treningowy wpływają na to, które funkcje zostaną wyuczone.

### Koszt obliczeń a liczba parametrów
Dla warstwy o szerokości $d$ projekcje Q, K i V mają po $d^2$ parametrów, projekcja wyjściowa ma $d^2$, a MLP $d\to 4d\to d$ ma $4d^2+4d^2$. Razem otrzymujemy około $12d^2$ wag na blok, bez biasów i norm. To przybliżenie nie jest liczbą parametrów całego modelu: trzeba dodać embeddingi, wszystkie warstwy, normy i ewentualne wagi współdzielone.

Koszt attention na sekwencji długości T jest rzędu $O(T^2d)$, podczas gdy projekcje i MLP kosztują około $O(Td^2)$. Zależność kwadratowa dotyczy par pozycji, a nie całego budżetu modelu w każdym zakresie T. Optymalizacje pamięciowe i algorytmy attention mogą ograniczyć przechowywanie macierzy wag, lecz nie zmieniają automatycznie wszystkich kosztów obliczeń ani jakości modelu.

### Dlaczego większy model nie oznacza automatycznie lepszego asystenta
Pretraining uczy przewidywania tekstu z korpusu, nie bezpośrednio „pomocności”. SFT uczy formatów instrukcja–odpowiedź. W podejściach RLHF ludzie porównują odpowiedzi, model nagrody aproksymuje te preferencje, a optymalizacja polityki przesuwa rozkład odpowiedzi. DPO bezpośrednio optymalizuje preferowane odpowiedzi w relacji do odpowiedzi odrzuconych, zwykle względem modelu referencyjnego. Te procedury mają różne założenia i nie gwarantują prawdziwości; preferencje nie są tym samym co poprawność faktów.

### Eksperyment maski jako test metodologii
Jeśli model podczas treningu widzi token celu w swoim kontekście, może osiągnąć małą stratę przez skrót, który jest niedostępny podczas generowania. Dlatego w GPT dopuszczamy w pozycji t tylko klucze j≤t. Porównanie z modelem bez maski ma ilustrować różnicę informacji dostępnej w obliczeniu, nie stanowić benchmarku jakości architektur: wynik zależy od syntetycznego zadania i szczegółów eksperymentu.

### GPT-3, GPT-4 i ChatGPT: poziomy opisu
GPT-3 odnosi się do rodziny modeli opisanej w publikacji, która raportowała wariant o 175 mld parametrów i badania few-shot. GPT-4 to późniejszy system, którego raport techniczny opisywał wyniki i ograniczenia, lecz nie ujawnił pełnych szczegółów architektury, danych i rozmiaru. ChatGPT opisuje doświadczenie asystenta konwersacyjnego i zmieniające się modele oraz elementy systemu. Należy oddzielać nazwę architektury, konkretną wersję modelu i produkt z jego narzędziami oraz politykami. Notebook zawiera trwałe pojęcia; przy potrzebie aktualnych wersji/API należy sprawdzać bieżącą dokumentację.

### Jak analizować model bez jednej mylącej metryki
Cross-entropy i perplexity oceniają przypisywanie prawdopodobieństwa sekwencjom. Benchmarks mierzą określony zestaw zadań i są wrażliwe na format promptu, dane treningowe oraz zanieczyszczenie. Ocena asystenta powinna łączyć testy kontrolowane, analizę błędów, kalibrację, ocenę przez ludzi i parametry operacyjne: opóźnienie, throughput, pamięć i koszt. Wynik należy interpretować w granicach konkretnego protokołu, nie jako uniwersalną miarę inteligencji.

## 1. Autoregresja i przeciek przyszłości
GPT modeluje $p(x_1,…,x_T)=∏_t p(x_t|x_{<t})$. Pretraining minimalizuje ujemny log-likelihood następnego tokenu. Bez maski model mógłby odczytać przyszłe tokeny. Poniższy eksperyment syntetyczny ilustruje problem; unmasked attention jest właściwe dla niektórych encoderów, ale nie dla GPT.

In [ ]:
import math,torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(4);device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
class TinyLM(nn.Module):
    def __init__(self,causal=True):
        super().__init__();self.causal=causal;self.e=nn.Embedding(16,32);self.qkv=nn.Linear(32,96);self.o=nn.Linear(32,16)
    def forward(self,x):
        B,T=x.shape;q,k,v=self.qkv(self.e(x)).chunk(3,-1);s=q@k.transpose(-2,-1)/math.sqrt(32)
        if self.causal:s=s.masked_fill(~torch.tril(torch.ones(T,T,dtype=torch.bool,device=x.device)),torch.finfo(s.dtype).min)
        return self.o(s.softmax(-1)@v)
seq=torch.tensor([[i%4 for i in range(20)] for _ in range(32)],device=device);x,y=seq[:,:-1],seq[:,1:];hist={}
for name,causal in [("causal",True),("bez maski",False)]:
    model=TinyLM(causal).to(device);opt=torch.optim.AdamW(model.parameters(),lr=.01);ls=[]
    for _ in range(100):
        z=model(x);loss=F.cross_entropy(z.reshape(-1,16),y.reshape(-1));opt.zero_grad();loss.backward();opt.step();ls.append(loss.item())
    hist[name]=ls
for n,v in hist.items():plt.plot(v,label=n)
plt.yscale("log");plt.xlabel("krok");plt.ylabel("cross entropy");plt.legend();plt.grid();plt.show()

## 2. Parametry i koszt
Dla $d=d_{model}$: QKV i wyjście attention to około $4d²$, MLP $d→4d→d$ to $8d²$, razem około $12d²$ parametrów na blok (plus biasy/normy). Wagi attention wymagają $O(T²)$ pamięci, a obliczenia $O(T²d)$. Podwojenie kontekstu zwiększa koszt tej części około czterokrotnie.

In [ ]:
def block_params(d):return 12*d*d+13*d
for d in (64,256,768,2048):print(d,f"~{block_params(d):,} parametrów/blok")
T=[128,256,512,1024,2048,4096]
plt.loglog(T,[(t/128)**2 for t in T],marker="o");plt.xlabel("długość T");plt.ylabel("koszt względny (T=128)");plt.title("Kwadratowa złożoność attention");plt.grid(True,which="both");plt.show()

## 3. Od bazowego GPT do asystenta
Typowy schemat (różni się między systemami): pretraining next-token na korpusie; supervised fine-tuning (SFT) na przykładach instrukcja–odpowiedź; dostrajanie preferencji, np. RLHF (model nagrody + optymalizacja polityki) lub DPO; integracja produktu z promptami, narzędziami, routingiem i filtrami. Pretraining nie gwarantuje posłuszeństwa instrukcjom, prawdziwości ani bezpieczeństwa.

## 4. GPT-3, GPT-4, ChatGPT

| Nazwa | Znaczenie | Co wiadomo / ograniczenia |
|---|---|---|
| GPT-3 | Model językowy opisany w 2020 r. | Publikacja opisała wariant do 175 mld parametrów i zdolności few-shot. |
| GPT-4 | Późniejszy model ogólnego przeznaczenia | Raport opisuje ewaluacje, lecz nie ujawnia wielu szczegółów architektury, danych i rozmiaru. |
| ChatGPT | Asystent konwersacyjny | Produkt/rodzina systemów dialogowych, nie synonim jednej architektury lub jednej wagi modelu. |

Nie przenoś liczby parametrów GPT-3 na GPT-4. Nazwa ChatGPT obejmuje zmieniające się systemy; ta część nie jest dokumentacją aktualnego API.

## 5. Analiza modelu i ćwiczenie
Analizuj poziom tokenu (logity, entropia, temperatura), sekwencji (loss, perplexity), zadania (jakość, kalibracja, odporność, wycieki) i systemu (opóźnienie, przepustowość, koszt, kontekst). Perplexity mierzy modelowanie tekstu, ale nie mierzy wprost faktograficzności, użyteczności dialogu ani bezpieczeństwa.

## Podsumowanie
GPT autoregresyjnie przewiduje tokeny. Maska causal blokuje przeciek, a standardowe attention rośnie kwadratowo względem długości kontekstu. Asystenci dodają dostrajanie oraz elementy systemowe.